In [ ]:
import requests
import json

def get_recommendation(model_name, prompt, system_prompt="Ты профессиональный ИТ-консультант.", temperature = 0.3):
    """
    Основной метод для получения ответа от локальной Ollama.
    """
    url = "http://localhost:11434/api/generate"
    
    # 1. Проверяем наличие модели в системе
    try:
        tags_response = requests.get("http://localhost:11434/api/tags")
        tags_response.raise_for_status()
        available_models = [m['name'] for m in tags_response.json().get('models', [])]
        
        # Проверяем точное совпадение или наличие тега :latest
        if model_name not in available_models and f"{model_name}:latest" not in available_models:
            return f"Ошибка: Модель '{model_name}' не найдена в Ollama. Сначала сделайте 'ollama pull {model_name}'"
            
    except requests.exceptions.ConnectionError:
        return "Ошибка: Ollama не запущена. Запустите приложение Ollama или сервис."

    # 2. Формируем запрос
    payload = {
        "model": model_name,
        "prompt": prompt,
        "system": system_prompt,
        "stream": False,  # Отключаем потоковую передачу для простоты MVP
        "options": {
            "temperature": temperature 
        }
    }

    # 3. Отправляем и возвращаем результат
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        return response.json().get('response', "Ошибка: Пустой ответ от модели.")
    except Exception as e:
        return f"Ошибка при запросе к модели: {str(e)}"


In [ ]:
FINANCIAL_MARKERS = {
    "revenue": ["выручка", "доход от реализации", "revenue", "sales"],
    "net_income": ["чистая прибыль", "убыток за период", "net income", "net profit"],
    "total_assets": ["итого активы", "совокупные активы", "total assets"],
    "equity": ["капитал", "собственный капитал", "total equity", "shareholders' equity"],
    "current_assets": ["оборотные активы", "current assets"],
    "current_liabilities": ["краткосрочные обязательства", "current liabilities"],
    "receivables": ["дебиторская задолженность", "accounts receivable", "receivables"],
    "cash_and_equivalents": ["денежные средства и их эквиваленты", "cash and cash equivalents"],
    "inventories": ["запасы", "inventories", "stock"],
    "operating_profit": ["операционная прибыль", "operating profit", "ebit"],
    "capex": ["капитальные вложения", "capex", "приобретение основных средств", "capital expenditure"],
    "cost_of_goods": ["себестоимость", "cost of sales", "cost of goods sold", "cogs"]
}

# Плоский список всех слов для быстрого поиска по страницам
all_keywords = [word for sublist in FINANCIAL_MARKERS.values() for word in sublist]

all_keywords

In [ ]:
import json

file_path = 'I:\\Projects\\AI-assistant\\output\\extracted_IFRS_12m2023_summary_all_20260321_195929.json' 

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
def get_filtered_content(data, keywords, top_n_pages=5):
    page_scores = []
    
    for i, page in enumerate(data.get('pages', [])):
        # Собираем весь текст страницы
        page_text = " ".join([b.get('content', '') for b in page.get('blocks', [])]).lower()
        
        # Считаем, сколько уникальных ключевых слов из нашего списка есть на странице
        score = sum(1 for word in keywords if word in page_text)
        
        # Игнорируем страницы, где обсуждают только руководство (как в твоем примере)
        if "вознаграждение" in page_text or "руководящего персонала" in page_text:
            score -= 5 
            
        if score > 0:
            page_scores.append((score, page_text))
    
    # Сортируем по релевантности (сначала самые "насыщенные" цифрами страницы)
    page_scores.sort(key=lambda x: x[0], reverse=True)
    
    # Берем топ-N страниц и склеиваем их
    final_text = "\n--- NEXT PAGE ---\n".join([p[1] for p in page_scores[:top_n_pages]])
    return final_text

# Использование:
content_text = get_filtered_content(data, all_keywords)
content_text


In [ ]:
import json

system_prompt = (
    "Ты финансовый аналитик. Извлеки значения за 2023 год для следующих показателей: "
    f"{', '.join(FINANCIAL_MARKERS.keys())}. "
    "Используй предоставленный текст финансовых таблиц. "
    "Если значение указано в миллионах, переведи в целое число (например, 1 803 млн -> 1803000000). "
    "Ответ дай СТРОГО в формате JSON. Если показателя нет, не включай его."
)

result = get_recommendation("qwen3.5:9b", f"{content_text}", system_prompt=system_prompt, temperature=0.1)

print(result)

In [ ]:
result

In [ ]:
import requests
import time

API_KEY = "sk-or-v1-..."

url = "https://openrouter.ai/api/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "HTTP-Referer": "http://localhost",
    "X-Title": "fallback-app"
}

# список моделей (от лучшей к худшей)
MODELS = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "meta-llama/llama-3.1-8b-instruct:free",
    "nousresearch/nous-hermes-2-mixtral-8x7b-dpo",
    "openchat/openchat-7b",
    "mistralai/mixtral-8x7b-instruct",
]

def ask_llm(messages, max_retries_per_model=2):
    for model in MODELS:
        print(f"\nПробуем модель: {model}")

        for attempt in range(max_retries_per_model):
            try:
                data = {
                    "model": model,
                    "messages": messages,
                    "temperature": 0.7,
                    "max_tokens": 300
                }

                response = requests.post(url, headers=headers, json=data)
                result = response.json()

                # успех
                if "choices" in result:
                    print(f"✅ Успех на модели: {model}")
                    return result["choices"][0]["message"]["content"]

                # ошибка API
                else:
                    error_msg = result.get("error", {}).get("message", "Unknown error")
                    print(f"❌ Ошибка: {error_msg}")

                    # если это rate limit → подождать и попробовать ещё раз
                    if response.status_code == 429:
                        time.sleep(2)
                    else:
                        break  # идём к следующей модели

            except Exception as e:
                print(f"⚠️ Exception: {e}")
                break

    return "❌ Все модели недоступны"

# ===== использование =====

messages = [
    {"role": "user", "content": "Привет! Объясни кратко что такое трансформеры"}
]

answer = ask_llm(messages)
print("\nОтвет:\n", answer)

In [ ]:
API_KEY = os.getenv("OPENROUTER_API_KEY") or "sk-or-..."

FILE_PATH = 'I:\\Projects\\AI-assistant\\output\\extracted_IFRS_12m2023_summary_all_20260321_195929.json'    # <<< меняешь сюда

COEFFICIENTS = ["revenue", "net_income", "total_assets", "equity", "current_assets", "current_liabilities", "receivables", "cash_and_equivalents", "inventories", "operating_profit", "capex", "cost_of_goods"]

In [ ]:
import requests
import time
import json
import os
import re

# ================= НАСТРОЙКИ =================

API_KEY = os.getenv("OPENROUTER_API_KEY") or "sk-or-v1-..."

FILE_PATH = 'I:\\Projects\\AI-assistant\\output\\extracted_IFRS_12m2023_summary_all_20260321_195929.json'   # <<< путь к файлу

COEFFICIENTS = ["revenue", "net_income", "total_assets", "equity", "current_assets", "current_liabilities", "receivables", "cash_and_equivalents", "inventories", "operating_profit", "capex", "cost_of_goods"]

MODELS = [
    "mistralai/mixtral-8x7b-instruct"
]

CHUNK_SIZE = 6000
MAX_RETRIES = 2

# ============================================


def load_text(file_path):
    ext = file_path.lower().split(".")[-1]

    if ext == "txt":
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    elif ext == "json":
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return json.dumps(data, ensure_ascii=False, indent=2)

    elif ext == "pdf":
        from pypdf import PdfReader
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += (page.extract_text() or "") + "\n"
        return text

    else:
        raise ValueError("Неподдерживаемый формат файла")


def split_text(text, chunk_size):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]


def safe_json_load(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}|\[.*\]', text, re.S)
        if match:
            try:
                return json.loads(match.group())
            except:
                return {}
        return {}


def ask_llm(messages):
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "coef-extractor"
    }

    for model in MODELS:
        print(f"\nПробуем модель: {model}")

        for attempt in range(MAX_RETRIES):
            try:
                data = {
                    "model": model,
                    "messages": messages,
                    "temperature": 0,
                    "max_tokens": 500
                }

                response = requests.post(url, headers=headers, json=data)
                result = response.json()

                if "choices" in result:
                    print("✅ успех")
                    return result["choices"][0]["message"]["content"]

                else:
                    error_msg = result.get("error", {}).get("message", "")
                    print("❌", error_msg)

                    if response.status_code == 429:
                        time.sleep(2)
                    else:
                        break

            except Exception as e:
                print("⚠️ exception:", e)
                break

    return None


def extract_from_chunk(chunk):
    prompt = f"""
Извлеки коэффициенты: {COEFFICIENTS}

Верни строго JSON ОБЪЕКТ:
{{"name": value}}

Запрещено:
- список []
- текст
- пояснения

Если нет значения → null

Текст:
{chunk}
"""

    messages = [{"role": "user", "content": prompt}]
    response = ask_llm(messages)

    if not response:
        return {}

    return safe_json_load(response)


def normalize_result(result):
    if isinstance(result, dict):
        return result

    elif isinstance(result, list):
        merged = {}
        for obj in result:
            if isinstance(obj, dict):
                merged.update(obj)
        return merged

    else:
        print("⚠️ неожиданный формат:", type(result))
        return {}


def main():
    text = load_text(FILE_PATH)
    chunks = split_text(text, CHUNK_SIZE)

    print(f"Всего чанков: {len(chunks)}")

    final_result = {k: None for k in COEFFICIENTS}

    for i, chunk in enumerate(chunks):
        print(f"\n=== chunk {i+1}/{len(chunks)} ===")

        raw_result = extract_from_chunk(chunk)
        result = normalize_result(raw_result)

        for k, v in result.items():
            if k in final_result and v is not None:
                final_result[k] = v

    print("\n=== ИТОГ ===")
    print(json.dumps(final_result, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
import requests

PROVIDERS = [
    {
        "name": "google",
        "url": "https://generativelanguage.googleapis.com/v1/models",
        "headers": lambda key: {"x-goog-api-key": key},
        "parser": lambda data: [m["name"] for m in data.get("models", [])],
        "validate": lambda r, data: r.status_code == 200 and "models" in data,
    },
    {
        "name": "openai",
        "url": "https://api.openai.com/v1/models",
        "headers": lambda key: {"Authorization": f"Bearer {key}"},
        "parser": lambda data: [m["id"] for m in data.get("data", [])],
        "validate": lambda r, data: r.status_code == 200 and "data" in data,
    },
    {
        "name": "mistral",
        "url": "https://api.mistral.ai/v1/models",
        "headers": lambda key: {"Authorization": f"Bearer {key}"},
        "parser": lambda data: [m["id"] for m in data.get("data", [])],
        "validate": lambda r, data: r.status_code == 200 and "data" in data,
    },
    {
        # ⚠️ OpenRouter В КОНЦЕ!
        "name": "openrouter",
        "url": "https://openrouter.ai/api/v1/models",
        "headers": lambda key: {"Authorization": f"Bearer {key}"},
        "parser": lambda data: [m["id"] for m in data.get("data", [])],
        "validate": lambda r, data: r.status_code == 200 and "data" in data,
    },
]

def detect_and_get_models(api_key):
    for provider in PROVIDERS:
        try:
            response = requests.get(
                provider["url"],
                headers=provider["headers"](api_key),
                timeout=5
            )

            data = response.json()

            if provider["validate"](response, data):
                models = provider["parser"](data)

                # дополнительная защита от OpenRouter
                if models:
                    return provider["name"], models

        except Exception:
            continue

    return None, []


# ====== ИСПОЛЬЗОВАНИЕ ======
if __name__ == "__main__":
    #api_key = "sk-or-v1-..."
    api_key = "AI..."

    provider, models = detect_and_get_models(api_key)

    if provider:
        print(f"\nDetected provider: {provider}")
        print("Available models:")
        for m in models:
            print("-", m)
    else:
        print("Could not detect provider or invalid API key.")

In [ ]:
import google.generativeai as genai

API_KEY = "AI..."
genai.configure(api_key=API_KEY)

print("Доступные модели для генерации текста:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

In [ ]:
import google.generativeai as genai

API_KEY = "AI..."
genai.configure(api_key=API_KEY)

# Меняем Pro на Flash или Lite! У них бесплатный лимит НЕ ноль.
# Попробуем версию 2.5-flash, она точно должна пустить
model = genai.GenerativeModel('gemini-2.5-flash') 

prompt = "Привет! Как дела?"
print(f"Отправляю запрос к {model.model_name}...\n")

try:
    response = model.generate_content(prompt)
    print("Ответ модели:")
    print(response.text)
except Exception as e:
    print("Произошла ошибка:", e)

In [ ]:
#def run_google(prompt, key, model_name = 'gemini-2.5-flash'):
def run_google(prompt, key, model_name = 'gemini-2.5-flash'):
    import google.generativeai as genai

    genai.configure(api_key=key)

    model = genai.GenerativeModel(model_name) 

    print(f"Отправляю запрос к {model.model_name}...\n")

    try:
        response = model.generate_content(prompt)
        print("Ответ модели:")
        return response.text
    except Exception as e:
        print("Произошла ошибка:", e)

run_google("Привет! Как дела?", "gemini_key")

In [ ]:
from groq import Groq

client = Groq(api_key="gsk_...")

completion = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {
            "role": "user",
            "content": "Привет! Расскажи о себе."
        }
    ],
    temperature=0.6,
    max_tokens=4096,
    top_p=0.95,
    stream=True,
)

for chunk in completion:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")

In [ ]:
#def run_groq(prompt, key, model_name = "qwen/qwen3-32b"):
def run_groq(prompt, key, model_name = "qwen/qwen3-32b"):
    from groq import Groq

    client = Groq(api_key=key)

    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.6,
        max_tokens=4096,
        top_p=0.95,
        stream=True,
    )

    content_parts = []
    for chunk in completion:
        if chunk.choices[0].delta.content:
            #print(chunk.choices[0].delta.content, end="")
            content_parts.append(chunk.choices[0].delta.content)

    return "".join(content_parts)

run_groq("Привет! Расскажи о себе.", "gsk_...")

In [ ]:
def get_available_models(model_family):
    if model_family == "Groq":
        return ["qwen/qwen3-32b"]
    if model_family == "google":
        return ['gemini-2.5-flash']
    if model_family == 'openrouter':
        return ['mistralai/mixtral-8x7b-instruct']

In [ ]:
def run_groq(prompt, key, model_name = "qwen/qwen3-32b"):
    from groq import Groq

    client = Groq(api_key=key)

    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.6,
        max_tokens=4096,
        top_p=0.95,
        stream=True,
    )

    for chunk in completion:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="")

def run_google(prompt, key, model_name = 'gemini-2.5-flash'):
    import google.generativeai as genai

    genai.configure(api_key=key)

    model = genai.GenerativeModel(model_name) 

    print(f"Отправляю запрос к {model.model_name}...\n")

    try:
        response = model.generate_content(prompt)
        print("Ответ модели:")
        print(response.text)
    except Exception as e:
        print("Произошла ошибка:", e)

def run_openrouter(prompt, key, model_name = "mistralai/mixtral-8x7b-instruct"):
    import requests
    import time

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "fallback-app"
    }

    try:
        data = {
            "model": model_name,
            "messages": [
                {
                    "role": "user", 
                    "content": prompt
                }
                ],
            "temperature": 0.7,
            "max_tokens": 300
        }

        response = requests.post(url, headers=headers, json=data)
        result = response.json()

        if "choices" in result:
            print("✅ Успех")
            return result["choices"][0]["message"]["content"]

        else:
            error_msg = result.get("error", {}).get("message", "Unknown error")
            print(f"❌ Ошибка: {error_msg}")

    except Exception as e:
        print(f"⚠️ Exception: {e}")


In [ ]:
def api_request(prompt, key, model_family, model_name):
    if model_family == "Groq":
        run_groq(prompt, key, model_name)
    if model_family == "google":
        run_google(prompt, key, model_name)
    if model_family == 'openrouter':
        run_openrouter(prompt, key, model_name)
